# Topic 09 — Solutions: Apply / Map / Transform

*Reference solutions for the objective tasks. Try the practice first.* The Warm-Up concept questions, Interview Lens and Reflection have **no key** — by design.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 30)
RAW = '../datasets/raw/'
products = pd.read_csv(RAW+'products.csv', dtype={'product_id':str})
products['list_price'] = np.where(products['list_price']<0, np.nan, products['list_price'])
orders = pd.read_csv(RAW+'orders.csv', dtype={'order_id':str,'customer_id':str})
orders['channel'] = orders['channel'].str.strip().str.title()
orders['status'] = orders['status'].str.strip().str.lower()
orders['order_date'] = pd.to_datetime(orders['order_date'], format='mixed', dayfirst=True, errors='coerce')
orders['month'] = orders['order_date'].dt.to_period('M').astype(str)
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
items['line_revenue'] = items['quantity']*items['unit_price']*(1-items['discount'])
master = (items.merge(products[['product_id','category','unit_cost']], on='product_id', how='left')
              .merge(orders[['order_id','channel','status','month','order_date']], on='order_id', how='left'))
master['line_cogs'] = master['quantity']*master['unit_cost']
master['line_profit'] = master['line_revenue'] - master['line_cogs']
import time
customers = pd.read_csv(RAW+'customers.csv', dtype={'customer_id':str}).drop_duplicates('customer_id')


### Warm-Up — NumPy recall (self-check)

In [ ]:
q = np.array([3,12,150,8,99])
bands = np.select([q>=100, q>=10], ['bulk','medium'], default='small')
print(list(bands))

### Core lesson tasks

In [ ]:
seg = {'consumer':'B2C','business':'B2B','pro athlete':'B2C'}
customers['seg2'] = customers['segment'].str.lower().map(seg)
print(customers['seg2'].value_counts(dropna=False))

In [ ]:
master['margin_band'] = np.where(master['line_profit'] > 0, 'profit', 'loss')
master['size_band'] = np.select([master['quantity']>=100, master['quantity']>=10], ['bulk','medium'], default='small')
print(master['size_band'].value_counts())

In [ ]:
v = master['quantity']*master['unit_price']*(1-master['discount'])
a = master.apply(lambda r: r['quantity']*r['unit_price']*(1-r['discount']), axis=1)
print(np.allclose(v, a, equal_nan=True))

### Mixed review (earlier topics)

In [ ]:
ship = pd.read_csv(RAW+'shipments.csv', dtype={'order_id':str}, parse_dates=['promised_date','shipped_date'])
ship['is_late'] = ship['shipped_date'] > ship['promised_date']
print(ship['is_late'].sum())

In [ ]:
print(master.groupby('size_band')['line_profit'].mean())

### Data detective

In [ ]:
ship = pd.read_csv(RAW+'shipments.csv', dtype={'order_id':str}, parse_dates=['promised_date','shipped_date'])
ship['is_late'] = ship['shipped_date'] > ship['promised_date']
print(ship['is_late'].mean())

### Interview Lens — discussion points (not full answers)
- **Q26:** vectorized ops run one compiled C loop over contiguous arrays; iterrows/apply(axis=1) build a Python object per row.
- **Q27:** apply is a smell when the body is plain arithmetic/conditionals expressible with column math, np.where, np.select, or map.